In [0]:
%run "./config"

In [0]:
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

In [0]:
dbutils.widgets.text('Date','')
input_value= dbutils.widgets.get('Date')

In [0]:
from datetime import date
today = date.today().strftime("%Y-%m-%d")
df = spark.read.option('header', 'true').csv(
  f'{raw_folder_path}/salesdata {today}/customer*'
)
df.show(2)

+----------+---------+-----+---------+----------+--------+------+------------------+--------------------+--------------------+------------+--------------------+------------+--------------------+--------------------+
|CustomerID|NameStyle|Title|FirstName|MiddleName|LastName|Suffix|       CompanyName|         SalesPerson|        EmailAddress|       Phone|        PasswordHash|PasswordSalt|             rowguid|        ModifiedDate|
+----------+---------+-----+---------+----------+--------+------+------------------+--------------------+--------------------+------------+--------------------+------------+--------------------+--------------------+
|         1|    FALSE|  Mr.|  Orlando|        N.|     Gee|  NULL|      A Bike Store|adventure-works\p...|orlando0@adventur...|245-555-0173|L/Rlwxzp4w7RWmEgX...|    1KjXYs4=|3f5ae95e-b87d-4ae...|2005-08-01T00:00:...|
|         2|    FALSE|  Mr.|    Keith|      NULL|  Harris|  NULL|Progressive Sports|adventure-works\d...|keith0@adventure-...|170-555-01

In [0]:

from pyspark.sql.functions import col,concat_ws, current_timestamp, lit, split
df_modified= df.withColumn('CustomerName', concat_ws('',col ("Title"), lit(' '), col("FirstName"),lit(' '), col("MiddleName"),lit(' '), col 
                ("LastName")))\
               .drop('PasswordHash','NameStyle','PasswordSalt','rowguid','ModifiedDate','Title','FirstName','LastName','MiddleName','Suffix')\
               .withColumn("Date",lit(input_value))

In [0]:
customer_data_processed= df_modified.write.mode( "append").option("header", "true").csv(f"{processed_folder_path}/customerdata {today}")